In [ ]:
pip install "gymnasium[classic-control]" stable-baselines3 sb3-contrib pyyaml pydantic

In [ ]:
pip uninstall -y torch torchvision torchaudio torchdata torchtext dgl

In [ ]:
pip install torch==2.1.2 

In [ ]:
pip install torchdata==0.7.1

In [ ]:
pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html

In [ ]:
pip install "numpy<2.0.0"

In [ ]:
pip install holidays

In [21]:
import datetime
import json
import numpy as np
import pandas as pd
import gymnasium as gym
from pathlib import Path

DEMAND = 0
CAPACITY = 1

def _build_special_days(year, n_days):
    import holidays as hl
    pt_holidays = hl.country_holidays("PT", years=[year])
    start = datetime.date(year, 1, 1)
    special = set()
    for d in range(n_days):
        date = start + datetime.timedelta(days=d)
        if date.weekday() == 6 or date in pt_holidays:
            special.add(d)
    return special


class ScheduleEnv(gym.Env):
    def __init__(self,
                 data_dir: str = "../../../../data/problems/SMARTASK_SIMPLE_2025",
                 capacity_slack: int = 1):
        super().__init__()
        base = Path(data_dir)

        with open(base / "problem.json") as f:
            prob = json.load(f)

        self.num_days = prob["temporalScope"]["numDays"]
        self.year = prob["temporalScope"]["year"]
        employees = prob["employees"]["simple"]
        self.num_employees = len(employees)
        self.employee_teams = [set(emp.get("teams", [])) for emp in employees]
        self.dual_team = [len(teams) > 1 for teams in self.employee_teams]

        shifts_sorted = sorted(prob["demand"]["shifts"], key=lambda s: s["order"])
        self.shift_codes = [s["code"] for s in shifts_sorted]
        self.shift_idx = {c: i for i, c in enumerate(self.shift_codes)}
        self.shift_order_map = {s["code"]: s["order"] for s in shifts_sorted}
        self.teams = list(prob["demand"]["organizationalUnits"]["teams"])
        self.team_idx = {t: i for i, t in enumerate(self.teams)}
        self.num_shifts = len(self.shift_codes)
        self.num_teams = len(self.teams)

        self.team_sizes = {
            team: sum(1 for s in self.employee_teams if team in s)
            for team in self.teams
        }

        vac_df = pd.read_csv(base / "vacations.csv", header=None)
        self.vac_mask = vac_df.iloc[:, 1:].values.astype(bool)

        dem_df = pd.read_csv(base / "demand.csv")
        dem_df["date"] = pd.to_datetime(dem_df["date"])
        start_ts = pd.Timestamp(f"{self.year}-01-01")
        dem_df["day_idx"] = (dem_df["date"] - start_ts).dt.days

        self.min_demand = np.zeros((self.num_days, self.num_shifts, self.num_teams), dtype=int)
        self.ideal_demand = np.zeros((self.num_days, self.num_shifts, self.num_teams), dtype=int)
        for _, row in dem_df.iterrows():
            d = int(row["day_idx"])
            s = self.shift_idx[row["shift"]]
            t = self.team_idx[row["team"]]
            self.min_demand[d, s, t] = int(row["minimum"])
            self.ideal_demand[d, s, t] = int(row["ideal"])

        self.special_days = _build_special_days(self.year, self.num_days)

        self.max_days_per_year = 223
        self.max_consecutive_days = 5
        self.special_days_cap = 22
        self.capacity_slack = capacity_slack

        self.PASS_ACTION = self.num_employees
        self.action_space = gym.spaces.Discrete(self.num_employees + 1)
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)

        self.reset()

    def build_slot_queue(self, min_demand, capacity_slack=1):
        num_days, num_shifts, num_teams = min_demand.shape
        demand_slots = []
        for d in range(num_days):
            for s in range(num_shifts):
                for t in range(num_teams):
                    headcount = int(min_demand[d, s, t])
                    for _ in range(headcount):
                        demand_slots.append((d, s, t, DEMAND))

        capacity_slots = []
        for d in range(num_days):
            for s in range(num_shifts):
                for t in range(num_teams):
                    deficit = int(self.ideal_demand[d, s, t] - min_demand[d, s, t])
                    for _ in range(deficit):
                        capacity_slots.append((d, s, t, CAPACITY))

        for _ in range(capacity_slack):
            for d in range(num_days):
                for s in range(num_shifts):
                    for t in range(num_teams):
                        capacity_slots.append((d, s, t, CAPACITY))

        return demand_slots + capacity_slots, len(demand_slots)

    def _get_assigned_shift(self, emp, day):
        s = self.emp_day_shift[emp, day]
        if s < 0:
            return None
        return self.shift_codes[s]

    def _get_prev_shift(self, emp, day):
        if day == 0:
            return None
        return self._get_assigned_shift(emp, day - 1)

    def _get_next_shift(self, emp, day):
        if day >= self.num_days - 1:
            return None
        return self._get_assigned_shift(emp, day + 1)

    def _consecutive_streak_if_work(self, emp, day):
        streak = 1
        d = day - 1
        while d >= 0 and self.emp_day_shift[emp, d] >= 0:
            streak += 1
            d -= 1
        d = day + 1
        while d < self.num_days and self.emp_day_shift[emp, d] >= 0:
            streak += 1
            d += 1
        return streak

    def _get_info(self):
        return {}

    def _obs(self):
        return np.zeros(1, dtype=np.float32)

    def current_slot(self):
        if self.slot_idx >= len(self.slot_queue):
            return None
        return self.slot_queue[self.slot_idx]

    def current_day(self):
        s = self.current_slot()
        return s[0] if s is not None else 0

    def _team_balance_bonus(self, emp_id, team):
        if not self.dual_team[emp_id]:
            return 0.0
        sizes = {t: self.team_sizes[t] for t in self.employee_teams[emp_id]}
        min_size, max_size = min(sizes.values()), max(sizes.values())
        if min_size == max_size:
            return 0.0
        smaller = {t for t, s in sizes.items() if s == min_size}
        return (max_size - min_size) * 0.5 if team in smaller else 0.0

    def _calculate_reward(self, action, day, s_idx, t_idx, kind):
        if action == self.PASS_ACTION:
            return -5.0 if kind == DEMAND else 0.0
        team = self.teams[t_idx]
        bonus = self._team_balance_bonus(action, team)
        if kind == DEMAND:
            return 1.0 + 0.2 * bonus
        cov = self.daily_coverage[day, s_idx, t_idx]
        if cov <= self.ideal_demand[day, s_idx, t_idx]:
            return 1.0 + 0.1 * bonus
        return 0.2

    def _calculate_final_reward(self):
        shortfall = int(np.maximum(0, self.min_demand - self.daily_coverage).sum())
        self.ideal_shortfall = int(np.maximum(0, self.ideal_demand - self.daily_coverage).sum())
        days_short = int(np.maximum(0, self.max_days_per_year - self.days_worked).sum())
        reward = 200.0 if shortfall == 0 else 0.0
        reward -= 0.5 * self.ideal_shortfall
        reward -= 5.0 * days_short
        return reward

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.days_worked = np.zeros(self.num_employees, dtype=int)
        self.special_days_worked = np.zeros(self.num_employees, dtype=int)
        self.daily_coverage = np.zeros(
            (self.num_days, self.num_shifts, self.num_teams), dtype=np.float32
        )
        self.emp_day_shift = np.full((self.num_employees, self.num_days), -1, dtype=int)
        self.emp_day_team = np.full((self.num_employees, self.num_days), -1, dtype=int)
        self.slot_queue, self.num_demand_slots = self.build_slot_queue(self.min_demand, self.capacity_slack)
        self.slot_idx = 0
        self.demand_skips = 0
        self.ideal_shortfall = 0
        return self._obs(), self._get_info()

    def step(self, action):
        if self.slot_idx >= len(self.slot_queue):
            return self._obs(), 0.0, True, False, self._get_info()
        day, s_idx, t_idx, kind = self.slot_queue[self.slot_idx]
        action = int(action)

        if action != self.PASS_ACTION:
            self.emp_day_shift[action, day] = s_idx
            self.emp_day_team[action, day] = t_idx
            self.days_worked[action] += 1
            self.daily_coverage[day, s_idx, t_idx] += 1
            if day in self.special_days:
                self.special_days_worked[action] += 1
        elif kind == DEMAND:
            self.demand_skips += 1

        reward = self._calculate_reward(action, day, s_idx, t_idx, kind)
        self.slot_idx += 1
        terminated = self.slot_idx >= len(self.slot_queue)
        if terminated:
            reward += self._calculate_final_reward()
        return self._obs(), reward, terminated, False, self._get_info()

    def _emp_can_cover(self, emp_id, day_id, shift_id, team_id):
        if self.teams[team_id] not in self.employee_teams[emp_id]:
            return False
        if self.vac_mask[emp_id, day_id]:
            return False
        if self.emp_day_shift[emp_id, day_id] >= 0:
            return False
        if self.days_worked[emp_id] >= self.max_days_per_year:
            return False
        if day_id in self.special_days and self.special_days_worked[emp_id] >= self.special_days_cap:
            return False
        if self._consecutive_streak_if_work(emp_id, day_id) > self.max_consecutive_days:
            return False

        prev_shift = self._get_prev_shift(emp_id, day_id)
        next_shift = self._get_next_shift(emp_id, day_id)
        shift_code = self.shift_codes[shift_id]

        if shift_code == "M" and prev_shift == "T":
            return False
        if shift_code == "T" and next_shift == "M":
            return False

        return True

    def get_employee_mask(self):
        mask = np.zeros(self.num_employees + 1, dtype=bool)
        if self.slot_idx >= len(self.slot_queue):
            mask[self.PASS_ACTION] = True
            return mask
        day, s_idx, t_idx, kind = self.slot_queue[self.slot_idx]
        for e in range(self.num_employees):
            if self._emp_can_cover(e, day, s_idx, t_idx):
                mask[e] = True
        if kind == CAPACITY:
            mask[self.PASS_ACTION] = True
        elif not mask[:self.num_employees].any():
            mask[self.PASS_ACTION] = True
        return mask

    def action_label(self, emp, day):
        s = self.emp_day_shift[emp, day]
        if s < 0:
            return "-"
        t = self.emp_day_team[emp, day]
        return f"{self.shift_codes[s]}-{self.teams[t]}"

    def render(self):
        for emp in range(self.num_employees):
            schedule = [self.action_label(emp, day) for day in range(self.num_days)]
            print(f"Employee {emp + 1:2d}: {schedule}")

In [22]:
import dgl
import dgl.nn as dglnn
import torch
import numpy as np


def emp_feat_dim(env):
    return 2 + env.num_teams + 1


def day_feat_dim(env):
    return 2 * env.num_shifts * env.num_teams + 2


def build_graph(env):
    sources, destinations = [], []
    for emp in range(env.num_employees):
        for day in range(env.num_days):
            sources.append(emp)
            destinations.append(day)
    graph_data = {
        ("employee", "assigned_to", "day"): (torch.tensor(sources), torch.tensor(destinations)),
        ("day", "staffed_by", "employee"): (torch.tensor(destinations), torch.tensor(sources)),
    }
    g = dgl.heterograph(graph_data)
    update_graph_features(g, env)
    return g


def _static_feats(env):
    if not hasattr(env, "_static_feat_cache"):
        team_flags = np.zeros((env.num_employees, env.num_teams), dtype=np.float32)
        for t_idx, team in enumerate(env.teams):
            for emp in range(env.num_employees):
                team_flags[emp, t_idx] = float(team in env.employee_teams[emp])
        emp_pos = (np.arange(env.num_employees) / env.num_employees).astype(np.float32)
        special = np.array([float(d in env.special_days) for d in range(env.num_days)],
                           dtype=np.float32)
        day_pos = (np.arange(env.num_days) / env.num_days).astype(np.float32)
        env._static_feat_cache = (team_flags, emp_pos, special, day_pos)
    return env._static_feat_cache


def update_graph_features(g, env):
    team_flags, emp_pos, special, day_pos = _static_feats(env)
    n_teams = env.num_teams
    n_pairs = env.num_shifts * n_teams
    D = env.num_days

    emp_feats = np.empty((env.num_employees, emp_feat_dim(env)), dtype=np.float32)
    emp_feats[:, 0] = env.days_worked / 223.0
    cur_day = env.current_day()
    for emp in range(env.num_employees):
        emp_feats[emp, 1] = env._consecutive_streak_if_work(emp, cur_day) / 5.0
    emp_feats[:, 2:2 + n_teams] = team_flags
    emp_feats[:, 2 + n_teams] = emp_pos

    day_feats = np.empty((D, day_feat_dim(env)), dtype=np.float32)
    day_feats[:, :n_pairs] = (env.min_demand - env.daily_coverage).transpose(0, 2, 1).reshape(D, n_pairs)
    day_feats[:, n_pairs:2 * n_pairs] = (env.ideal_demand - env.daily_coverage).transpose(0, 2, 1).reshape(D, n_pairs)
    day_feats[:, 2 * n_pairs] = special
    day_feats[:, 2 * n_pairs + 1] = day_pos

    g.nodes["employee"].data["feat"] = torch.from_numpy(emp_feats)
    g.nodes["day"].data["feat"] = torch.from_numpy(day_feats)


def coverage_gap_vector(env, day_id):
    cov = env.daily_coverage[day_id]  # (S, T)
    min_gap = (env.min_demand[day_id] - cov).T.reshape(-1)
    ideal_gap = (env.ideal_demand[day_id] - cov).T.reshape(-1)
    return np.concatenate([min_gap, ideal_gap]).astype(np.float32)

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GNNActorCritic(nn.Module):
    def __init__(self, emp_in_feats, day_in_feats, num_shifts, num_teams,
                 hidden_dim=64, encoded_dim=64):
        super().__init__()
        self.encoded_dim = encoded_dim
        self.num_shifts = num_shifts
        self.num_teams = num_teams
        # min-gap + ideal-gap blocks, matching coverage_gap_vector
        self.cov_gap_dim = 2 * num_shifts * num_teams

        self.emp_proj = nn.Linear(emp_in_feats, hidden_dim)
        self.day_proj = nn.Linear(day_in_feats, hidden_dim)

        self.conv1 = dglnn.HeteroGraphConv({
            'assigned_to': dglnn.SAGEConv(hidden_dim, hidden_dim, 'mean'),
            'staffed_by':  dglnn.SAGEConv(hidden_dim, hidden_dim, 'mean'),
        }, aggregate='sum')

        self.conv2 = dglnn.HeteroGraphConv({
            'assigned_to': dglnn.SAGEConv(hidden_dim, encoded_dim, 'mean'),
            'staffed_by':  dglnn.SAGEConv(hidden_dim, encoded_dim, 'mean'),
        }, aggregate='sum')

        # 64 + 16 + 2 + 4 + 1 = 87
        ctx_dim = encoded_dim + self.cov_gap_dim + self.num_shifts + self.num_teams + 1
        head_in = encoded_dim + ctx_dim

        # Pointer score: one logit per employee
        self.score_head  = nn.Sequential(
            nn.Linear(head_in, 64), 
            nn.Tanh(), 
            nn.Linear(64, 1)
        )

        # PASS + value condition on a workforce summary (mean emp_emb) + slot ctx.
        self.pass_head   = nn.Sequential(
            nn.Linear(head_in, 64),
            nn.Tanh(), 
            nn.Linear(64, 1)
        )

        self.critic_head = nn.Sequential(
            nn.Linear(head_in, 64), 
            nn.Tanh(), 
            nn.Linear(64, 1)
        )

    @classmethod
    def from_env(cls, env, hidden_dim=64, encoded_dim=64):
        return cls(
            emp_in_feats=emp_feat_dim(env),
            day_in_feats=day_feat_dim(env),
            num_shifts=env.num_shifts,
            num_teams=env.num_teams,
            hidden_dim=hidden_dim, 
            encoded_dim=encoded_dim,
        )

    def gnn_forward(self, g):
        h = {
            "employee": self.emp_proj(g.nodes["employee"].data["feat"]),
            "day": self.day_proj(g.nodes["day"].data["feat"])
        }

        h = self.conv1(g, h); 
        h = {k: F.relu(v) for k, v in h.items()}
        h = self.conv2(g, h); 
        h = {k: F.relu(v) for k, v in h.items()}
        return h["employee"], h["day"]

    def _heads_multi(self, emp_emb_b, day_vec, cov_gaps, shift, team, kind, action_masks):
        B, E, _ = emp_emb_b.shape
        ctx = torch.cat([day_vec, cov_gaps / 10.0, shift, team, kind], dim=-1)   # (B, 87)

        ctx_b = ctx.unsqueeze(1).expand(B, E, -1)
        emp_logits = self.score_head(torch.cat([emp_emb_b, ctx_b], dim=-1)).squeeze(-1)  # (B, E)

        pooled = torch.cat([emp_emb_b.mean(1), ctx], dim=-1)
        pass_logit = self.pass_head(pooled)                                       # (B, 1)
        values = self.critic_head(pooled).squeeze(-1)                             # (B,)

        logits = torch.cat([emp_logits, pass_logit], dim=-1)                      # (B, E+1)
        masks_bool = torch.as_tensor(action_masks, dtype=torch.bool)
        if masks_bool.dim() == 1:
            masks_bool = masks_bool.unsqueeze(0)
        all_invalid = ~masks_bool.any(dim=-1)
        if all_invalid.any():
            masks_bool = masks_bool.clone()
            masks_bool[all_invalid, -1] = True
        logits = logits.masked_fill(~masks_bool, float("-inf"))
        return F.softmax(logits, dim=-1), values

    def _heads(self, emp_emb, day_emb, day_ids, cov_gaps, shift, team, kind, action_masks):
        # Single-snapshot case: the whole batch shares one embedding set.
        B = day_ids.shape[0]
        emp_emb_b = emp_emb.unsqueeze(0).expand(B, -1, -1)
        return self._heads_multi(emp_emb_b, day_emb[day_ids],
                                 cov_gaps, shift, team, kind, action_masks)

    def forward(self, g, day_ids, cov_gaps, shift, team, kind, action_masks):
        emp_emb, day_emb = self.gnn_forward(g)
        return self._heads(emp_emb, day_emb, day_ids, cov_gaps,
                           shift, team, kind, action_masks)

In [24]:
from dataclasses import dataclass, field
from torch.distributions import Categorical


@dataclass
class Trajectory:
    day_ids: list = field(default_factory=list)
    coverage_gaps: list = field(default_factory=list)
    shifts: list = field(default_factory=list)
    teams: list = field(default_factory=list)
    kinds: list = field(default_factory=list)
    action_masks: list = field(default_factory=list)
    actions: list = field(default_factory=list)
    log_probs_old: list = field(default_factory=list)
    rewards: list = field(default_factory=list)
    values: list = field(default_factory=list)
    snap_ids: list = field(default_factory=list)
    emp_feats_unique: list = field(default_factory=list)
    day_feats_unique: list = field(default_factory=list)


    def to_tensors(self):
        return {
            "day_ids": torch.tensor(self.day_ids, dtype=torch.long),
            "coverage_gaps": torch.stack(self.coverage_gaps),
            "shifts": torch.stack(self.shifts),
            "teams": torch.stack(self.teams),
            "kinds": torch.stack(self.kinds),
            "action_masks": torch.stack(self.action_masks),
            "actions": torch.tensor(self.actions, dtype=torch.long),
            "log_probs_old": torch.tensor(self.log_probs_old, dtype=torch.float32),
            "rewards": torch.tensor(self.rewards, dtype=torch.float32),
            "values": torch.tensor(self.values, dtype=torch.float32),
            "snap_ids": torch.tensor(self.snap_ids, dtype=torch.long),
            "emp_feats_unique": torch.stack(self.emp_feats_unique),
            "day_feats_unique": torch.stack(self.day_feats_unique),
        }


def collect_trajectory(env, model, graph):
    model.eval()
    traj = Trajectory()
    env.reset()
    terminated = truncated = False

    cached_day_id = None
    cached_emp_emb = None
    cached_day_emb = None
    cached_emp_feats = None
    cached_day_feats = None


    with torch.no_grad():
        while not (terminated or truncated):
            slot = env.current_slot()
            if slot is None:
                break
            day_id, s_idx, t_idx, kind = slot

            # GNN embeddings depend on per-day state → recompute only when the day changes.
            if day_id != cached_day_id:
                update_graph_features(graph, env)
                cached_emp_emb, cached_day_emb = model.gnn_forward(graph)
                cached_day_id = day_id
                traj.emp_feats_unique.append(graph.nodes["employee"].data["feat"].clone())
                traj.day_feats_unique.append(graph.nodes["day"].data["feat"].clone())
                cur_snap = len(traj.emp_feats_unique) - 1

            # Per-step inputs describing THIS slot.
            cov_gap = coverage_gap_vector(env, day_id)
            shift_oh = [0.0] * env.num_shifts; shift_oh[s_idx] = 1.0
            team_oh = [0.0] * env.num_teams;  team_oh[t_idx]  = 1.0
            mask = env.get_employee_mask()

            day_id_t = torch.tensor([day_id], dtype=torch.long)
            cov_gap_t = torch.from_numpy(cov_gap).unsqueeze(0)
            shift_t = torch.tensor([shift_oh], dtype=torch.float32)
            team_t = torch.tensor([team_oh], dtype=torch.float32)
            kind_t = torch.tensor([[float(kind)]], dtype=torch.float32)
            mask_t = torch.tensor(mask.tolist(), dtype=torch.bool).unsqueeze(0)

            probs, values = model._heads(
                cached_emp_emb, cached_day_emb,
                day_id_t, cov_gap_t, shift_t, team_t, kind_t, mask_t,
            )

            dist = Categorical(probs=probs[0])
            action = dist.sample()

            _, reward, terminated, truncated, _ = env.step(action.item())

            traj.day_ids.append(day_id)
            traj.coverage_gaps.append(cov_gap_t[0])
            traj.shifts.append(shift_t[0])
            traj.teams.append(team_t[0])
            traj.kinds.append(kind_t[0])
            traj.action_masks.append(mask_t[0])
            traj.actions.append(action.item())
            traj.log_probs_old.append(dist.log_prob(action).item())
            traj.rewards.append(float(reward))
            traj.values.append(values[0].item())
            traj.snap_ids.append(cur_snap)

    return traj

In [25]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    T = len(rewards)
    rewards_a = np.array(rewards, dtype=np.float32)
    values_a = np.array(values,  dtype=np.float32)
    values_ext = np.append(values_a, 0.0)

    advantages = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        delta = rewards_a[t] + gamma * values_ext[t + 1] - values_ext[t]
        gae = delta + gamma * lam * gae
        advantages[t] = gae

    returns = advantages + values_a
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-5)

    return (
        torch.tensor(advantages, dtype=torch.float32),
        torch.tensor(returns, dtype=torch.float32),
    )


In [26]:
from torch.distributions import Categorical


def ppo_update(model, optimizer, batch, advantages, returns, graph,
               clip_eps=0.2, value_coeff=0.5, entropy_coeff=0.01,
               K_epochs=4, mini_batch_size=512):
    T = batch["actions"].shape[0]
    snap_ids = batch["snap_ids"]
    U = int(snap_ids.max().item()) + 1
    E = graph.num_nodes("employee")
    D = graph.num_nodes("day")
    losses, track_entropy = [], []
    model.train()

    for _ in range(K_epochs):
        # Shuffle whole snapshot groups (keeps each day's steps together so a
        # minibatch touches only a handful of snapshots).
        rank = torch.randperm(U)
        order = torch.argsort(rank[snap_ids], stable=True)

        for start in range(0, T, mini_batch_size):
            idx = order[start:start + mini_batch_size]
            snap_mb = snap_ids[idx]
            # snaps: unique snapshot ids in this minibatch; local: for each
            # sample, the position of its snapshot within `snaps`.
            snaps, local = torch.unique(snap_mb, return_inverse=True)
            n = snaps.shape[0]

            # One batched graph holding all n snapshots as disjoint components.
            bg = dgl.batch([graph] * n)
            bg.nodes["employee"].data["feat"] = batch["emp_feats_unique"][snaps].reshape(n * E, -1)
            bg.nodes["day"].data["feat"] = batch["day_feats_unique"][snaps].reshape(n * D, -1)
            emp_emb, day_emb = model.gnn_forward(bg)
            emp_emb = emp_emb.view(n, E, -1)
            day_emb = day_emb.view(n, D, -1)

            probs, values = model._heads_multi(
                emp_emb[local],                          # (B, E, dim) — own snapshot each
                day_emb[local, batch["day_ids"][idx]],   # (B, dim)    — own snapshot's day
                batch["coverage_gaps"][idx],
                batch["shifts"][idx], batch["teams"][idx], batch["kinds"][idx],
                batch["action_masks"][idx],
            )

            dist = Categorical(probs=probs)
            logp = dist.log_prob(batch["actions"][idx])
            ratios = torch.exp(logp - batch["log_probs_old"][idx])
            adv = advantages[idx]
            surr1 = ratios * adv
            surr2 = torch.clamp(ratios, 1 - clip_eps, 1 + clip_eps) * adv
            actor_loss = -torch.min(surr1, surr2).mean() - entropy_coeff * dist.entropy().mean()
            critic_loss = F.smooth_l1_loss(values, returns[idx])
            loss = actor_loss + value_coeff * critic_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            losses.append(loss.item())
            track_entropy.append(dist.entropy().mean().item())

    return float(np.mean(losses)), float(np.mean(track_entropy))

In [28]:
import os

# Hyperparameters
NUM_EPISODES = 10000
GAMMA = 1.0   # finite-horizon episodic task: terminal reward must reach every decision undiscounted
LAM = 0.95
CLIP_EPS = 0.15
VALUE_COEFF = 0.5
ENTROPY_COEFF_0 = 0.01
ENTROPY_DECAY = 0.995
ENTROPY_MIN = 0.001
K_EPOCHS = 3
MINI_BATCH_SIZE = 512
LR = 1.5e-4
SAVE_EVERY = 50

BEST_CKPT = "best_2teams_v3_idealslots.pth"
LATEST_CKPT = "latest_2teams_v3_idealslots.pth"
RESUME_FROM = "latest_2teams_v3_idealslots.pth"

env = ScheduleEnv()
graph = build_graph(env)
# Model dims are derived from the env so any scenario size works without code changes.
model = GNNActorCritic.from_env(env)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, eps=1e-5)

best_reward = -float("inf")
start_episode = 0

if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    for group in optimizer.param_groups:
        group["lr"] = LR
    start_episode = ckpt["episode"]
    best_reward = float(ckpt.get("best_reward", -float("inf")))
    print(f"Resumed from episode {start_episode}, best reward = {best_reward:.1f}")

for episode in range(start_episode, NUM_EPISODES):
    entropy_coeff = max(ENTROPY_MIN, ENTROPY_COEFF_0 * (ENTROPY_DECAY ** episode))

    traj = collect_trajectory(env, model, graph)
    batch = traj.to_tensors()

    advantages, returns = compute_gae(traj.rewards, traj.values, GAMMA, LAM)

    mean_loss, mean_entropy = ppo_update(
        model, optimizer, batch, advantages, returns, graph,
        clip_eps=CLIP_EPS,
        value_coeff=VALUE_COEFF,
        entropy_coeff=entropy_coeff,
        K_epochs=K_EPOCHS,
        mini_batch_size=MINI_BATCH_SIZE,
    )

    total_reward = sum(traj.rewards)
    shortfall = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())
    ideal_gap = int(np.maximum(0, env.ideal_demand - env.daily_coverage).sum())
    days_short = int(np.maximum(0, env.max_days_per_year - env.days_worked).sum())
    demand_skips = env.demand_skips
    mean_days = env.days_worked.mean()

    if total_reward > best_reward:
        best_reward = float(total_reward)
        torch.save(
            {"episode": episode, "model_state_dict": model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(), "best_reward": best_reward},
            BEST_CKPT,
        )
        print(f"  NEW BEST: {best_reward:.1f} at ep {episode} "
              f"| shortfall={shortfall} | ideal_gap={ideal_gap} "
              f"| demand_skips={demand_skips} | days_short={days_short}")

    if episode % 10 == 0:
        print(f"Ep {episode:>4} | R={total_reward:>8.1f} | Best={best_reward:>8.1f} "
              f"| Loss={mean_loss:.4f} | ent={mean_entropy:.4f} (coeff={entropy_coeff:.4f}) "
              f"| shortfall={shortfall} | ideal_gap={ideal_gap} "
              f"| skips={demand_skips} | days_short={days_short} | days={mean_days:.0f}")

    if (episode + 1) % SAVE_EVERY == 0:
        torch.save(
            {"episode": episode + 1, "model_state_dict": model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(), "best_reward": best_reward},
            LATEST_CKPT,
        )

Resumed from episode 300, best reward = 2512.7


KeyboardInterrupt: 

In [29]:
def evaluate(model, env, graph, greedy=False):
    model.eval()
    env.reset()
    terminated = truncated = False
    total_reward = 0.0

    cached_day_id = None
    cached_emp_emb = None
    cached_day_emb = None

    with torch.no_grad():
        while not (terminated or truncated):
            slot = env.current_slot()
            if slot is None:
                break
            day_id, s_idx, t_idx, kind = slot

            if day_id != cached_day_id:
                update_graph_features(graph, env)
                cached_emp_emb, cached_day_emb = model.gnn_forward(graph)
                cached_day_id = day_id

            cov_gap = coverage_gap_vector(env, day_id)
            shift_oh = [0.0] * env.num_shifts; shift_oh[s_idx] = 1.0
            team_oh = [0.0] * env.num_teams;  team_oh[t_idx]  = 1.0
            mask = env.get_employee_mask()

            day_id_t = torch.tensor([day_id], dtype=torch.long)
            cov_gap_t = torch.from_numpy(cov_gap).unsqueeze(0)
            shift_t = torch.tensor([shift_oh], dtype=torch.float32)
            team_t = torch.tensor([team_oh], dtype=torch.float32)
            kind_t = torch.tensor([[float(kind)]], dtype=torch.float32)
            mask_t = torch.tensor(mask.tolist(), dtype=torch.bool).unsqueeze(0)

            probs, _ = model._heads(
                cached_emp_emb, cached_day_emb,
                day_id_t, cov_gap_t, shift_t, team_t, kind_t, mask_t,
            )

            action = probs[0].argmax() if greedy else Categorical(probs=probs[0]).sample()
            _, reward, terminated, truncated, _ = env.step(action.item())
            total_reward += reward

    shortfall = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())
    ideal_gap = int(np.maximum(0, env.ideal_demand - env.daily_coverage).sum())

    snapshot = {
        "demand_skips": env.demand_skips,
        "ideal_gap": ideal_gap,
        "daily_coverage": env.daily_coverage.copy(),
    }

    schedule = [
        [env.action_label(emp, day) for day in range(env.num_days)]
        for emp in range(env.num_employees)
    ]

    return total_reward, shortfall, env.days_worked.tolist(), schedule, snapshot

ckpt = torch.load(BEST_CKPT, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Checkpoint: episode {ckpt['episode']}, best reward = {ckpt['best_reward']:.1f}\n")

# Generate N schedules
N = 20
results = []
for i in range(N):
    torch.manual_seed(42 + i)  # sampling is torch-based, so seed torch (not numpy)
    reward, shortfall, days_worked, schedule, snap = evaluate(model, env, graph, greedy=False)
    results.append((reward, shortfall, days_worked, schedule, snap))
    print(f"Run {i+1:2d} | Reward: {reward:8.1f} | Shortfall: {shortfall:3d} "
          f"| ideal_gap: {snap['ideal_gap']:3d} "
          f"| demand_skips: {snap['demand_skips']:3d} "
          f"| Days worked: {days_worked} (avg={np.mean(days_worked):.0f})")

# Pick the run with the lowest shortfall, breaking ties on ideal_gap
best_idx = min(range(N), key=lambda i: (results[i][1], results[i][4]["ideal_gap"]))
best_reward, best_shortfall, best_days, best_schedule, best_snap = results[best_idx]

print(f"\n{'='*60}")
print(f"BEST: Run {best_idx+1} | Reward: {best_reward:.1f} | Shortfall: {best_shortfall} "
      f"| Ideal gap: {best_snap['ideal_gap']}")
print(f"Days worked/employee: {best_days} (avg={np.mean(best_days):.0f})")
print(f"{'='*60}\n")

# Per-day shortfall breakdown for the best run, naming every (shift, team) gap.
best_coverage = best_snap["daily_coverage"]
for d in range(env.num_days):
    gaps = []
    for s_idx, shift in enumerate(env.shift_codes):
        for t_idx, team in enumerate(env.teams):
            gap = int(env.min_demand[d, s_idx, t_idx] - best_coverage[d, s_idx, t_idx])
            if gap > 0:
                gaps.append(f"{shift}-{team}={gap}")
    if gaps:
        print(f"Day {d:3d}: {', '.join(gaps)}")

print()
for emp in range(len(best_schedule)):
    print(f"Employee {emp+1:2d}: {best_schedule[emp]}")

import csv
schedule_csv = f"best_schedule_{env.num_teams}teams_v3.csv"
with open(schedule_csv, "w", newline="") as f:
    writer = csv.writer(f)
    header = ["Employee"] + [f"Day_{d}" for d in range(len(best_schedule[0]))]
    writer.writerow(header)
    for emp in range(len(best_schedule)):
        writer.writerow([f"Employee_{emp+1}"] + best_schedule[emp])

print(f"\nBest schedule written to {schedule_csv}")

Checkpoint: episode 214, best reward = 2512.7

Run  1 | Reward:   2509.4 | Shortfall:  18 | ideal_gap: 188 | demand_skips:  18 | Days worked: [223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223] (avg=223)
Run  2 | Reward:   2502.1 | Shortfall:  18 | ideal_gap: 183 | demand_skips:  18 | Days worked: [223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223] (avg=223)
Run  3 | Reward:   2513.0 | Shortfall:  18 | ideal_gap: 185 | demand_skips:  18 | Days worked: [223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223] (avg=223)
Run  4 | Reward:   2523.7 | Shortfall:  17 | ideal_gap: 177 | demand_skips:  17 | Days worked: [223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223] (avg=223)
Run  5 | Reward:   2512.2 | Shortfall:  17 | ideal_gap: 187 | demand_skips:  17 | Days worked: [223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223] (avg=223)
Run  6 | Reward:   2510.8 | Shortfall:  18 | ideal_gap: 185 | demand_skips:  18 | Days worked: [223, 223, 223, 223, 223, 22

In [ ]:
import matplotlib.pyplot as plt

ckpt = torch.load(BEST_CKPT, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
_, _, days_worked_list, _, snap = evaluate(model, env, graph, greedy=False)

days   = np.arange(env.num_days)
cov    = snap["daily_coverage"]   # (num_days, num_shifts, num_teams)
demand = env.min_demand           # (num_days, num_shifts, num_teams)

# Coverage-vs-demand grid: one subplot per (shift, team) pair.
fig, axes = plt.subplots(
    env.num_shifts, env.num_teams,
    figsize=(4 * env.num_teams, 4 * env.num_shifts),
    sharex=True, sharey=True, squeeze=False,
)
for s_idx, shift in enumerate(env.shift_codes):
    for t_idx, team in enumerate(env.teams):
        ax = axes[s_idx][t_idx]
        ax.plot(days, cov[:, s_idx, t_idx], label='Coverage', alpha=0.7)
        ax.plot(days, demand[:, s_idx, t_idx], label='Demand', alpha=0.7, linestyle='--', color='r')
        ax.fill_between(
            days, cov[:, s_idx, t_idx], demand[:, s_idx, t_idx],
            where=cov[:, s_idx, t_idx] < demand[:, s_idx, t_idx],
            color='red', alpha=0.2, label='Shortfall',
        )
        ax.set_title(f"{shift}-{team}")
        if t_idx == 0:
            ax.set_ylabel('Employees')
        if s_idx == env.num_shifts - 1:
            ax.set_xlabel('Day of year')
        ax.legend(loc='upper right', fontsize=8)

fig.suptitle('Coverage vs Demand across the year', fontsize=14)
plt.tight_layout()
plt.show()

# Quarterly violation breakdown — one column per (shift, team) pair plus a total.
pair_labels = [f"{s}-{t}" for t in env.teams for s in env.shift_codes]
header = f"{'Quarter':<12} " + " ".join(f"{p:>8}" for p in pair_labels) + f" {'Total':>6}"
print()
print(header)
print("-" * len(header))
total_viol = 0
for q, (start, end) in enumerate([(0, 91), (91, 182), (182, 273), (273, 365)], 1):
    per_pair = []
    q_total = 0
    for team in env.teams:
        for shift in env.shift_codes:
            s_idx, t_idx = env.shift_idx[shift], env.team_idx[team]
            v = int(np.sum(np.maximum(0, demand[start:end, s_idx, t_idx] - cov[start:end, s_idx, t_idx])))
            per_pair.append(v)
            q_total += v
    total_viol += q_total
    print(f"Q{q} ({start:3d}-{end:3d})  " + " ".join(f"{v:>8}" for v in per_pair) + f" {q_total:>6}")
print("-" * len(header))
print(f"{'Total':<12} " + " ".join(f"{'':>8}" for _ in pair_labels) + f" {total_viol:>6}")

days_worked_arr = np.array(days_worked_list)
print(f"\nDays worked/employee: {days_worked_list}")
print(f"Mean: {days_worked_arr.mean():.1f}, Min: {days_worked_arr.min()}, Max: {days_worked_arr.max()}")
